In [8]:
import ee
import geemap
import ipywidgets as widgets
from ipyleaflet import WidgetControl, GeoJSON
import math
import json
import re
import time
import functools
import traceback
import unicodedata
import pandas as pd
import osmnx as ox
import matplotlib.pyplot as plt
from shapely.geometry import shape, mapping, box

# ============================================================
# 1. INICIO DE GOOGLE EARTH ENGINE
# ============================================================
PROYECTO = 'booming-list-511608-s2'
try:
    ee.Initialize(project=PROYECTO)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROYECTO)


# ============================================================
# 2. MAPA (con arreglo del error 'east ... None')
# ============================================================
class MapaSeguro(geemap.Map):
    """Mapa que ignora los limites vacios (None) que manda el navegador."""
    def set_state(self, sync_data):
        claves_limites = ('north', 'south', 'east', 'west', 'bounds')
        limpio = {k: v for k, v in sync_data.items()
                  if not (k in claves_limites and v is None)}
        super().set_state(limpio)


Map = MapaSeguro(center=[22.58, -102.52], zoom=13, height='680px')

# ============================================================
# 3. CONFIGURACION GENERAL Y FUENTES DE DATOS
# ============================================================
# --- Sentinel-2 ---
FECHA_INI, FECHA_FIN = '2026-01-01', '2026-10-01'
BANDAS_S2 = ['B2', 'B3', 'B4', 'B8', 'B11']        # solo las que usan los indices

# --- Topografia (SRTM 30 m) ---
DEM = ee.Image('USGS/SRTMGL1_003')

# --- Limites administrativos (FAO GAUL 2015, dentro de GEE) ---
GAUL_PAISES = ee.FeatureCollection('FAO/GAUL/2015/level0')
GAUL_ESTADOS = ee.FeatureCollection('FAO/GAUL/2015/level1')
GAUL_MUNICIPIOS = ee.FeatureCollection('FAO/GAUL/2015/level2')

# --- Limites de OSM ---
MAX_KM2 = 25        # Area maxima para cargar OSM
MAX_DIBUJO = 15000  # Limite de estructuras dibujadas
MAX_MUNICIPIOS = 700

# --- Bandas y escalas de exportacion ---
INDICES_ESPECTRALES = ['NDVI', 'NDWI', 'MNDWI', 'EVI', 'NDSI', 'BSI', 'SAVI', 'NDMI']
BANDAS_TOPO = ['ELEVACION', 'PENDIENTE']
TODAS_LAS_BANDAS = INDICES_ESPECTRALES + BANDAS_TOPO
ESCALAS = [10, 20, 30, 60, 100, 250, 500, 1000, 2000, 5000]   # metros
FACTOR_ERROR_POS = math.sqrt(2) / 2     # semidiagonal de la celda: error posicional maximo
LIMITE_FILAS_TOTAL = 100_000_000


def construir_base(region=None):
    """Mediana de Sentinel-2 (solo las bandas necesarias), limitada a la region."""
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterDate(FECHA_INI, FECHA_FIN)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 15)))
    if region is not None:
        col = col.filterBounds(region)
    return col.select(BANDAS_S2).median().divide(10000)


# ============================================================
# 4. MENSAJES VISIBLES EN EL PANEL + PROTECCION DE ERRORES
# ============================================================
salida = widgets.Output(layout=widgets.Layout(
    height='120px', width='100%', overflow='auto', border='1px solid #ddd'))


def msg(*partes):
    with salida:
        print(*partes)


def seguro(funcion):
    """Si un boton falla, el error se ve en el panel en lugar de perderse."""
    @functools.wraps(funcion)
    def envoltura(*args, **kwargs):
        try:
            return funcion(*args, **kwargs)
        except Exception as ex:
            msg(f"ERROR en {funcion.__name__}: {ex}")
            with salida:
                traceback.print_exc()
    return envoltura


# ============================================================
# 5. WIDGETS (controles de la interfaz)
# ============================================================
estilo = {'description_width': '95px'}
ancho_total = widgets.Layout(width='100%')


def boton(texto, estilo_boton, ancho='auto'):
    return widgets.Button(description=texto, button_style=estilo_boton,
                          layout=widgets.Layout(width=ancho, flex='1 1 0', margin='2px'))


def titulo(texto):
    return widgets.HTML(f"<b style='color:#444'>{texto}</b>")


# --- Analisis ---
index_selector = widgets.Dropdown(
    options=[
        ('NDVI (Indice de Vegetacion Normalizado)', 'NDVI'),
        ('NDWI (Indice de Agua Normalizado)', 'NDWI'),
        ('MNDWI (Indice de Agua Modificado)', 'MNDWI'),
        ('EVI (Indice de Vegetacion Mejorado)', 'EVI'),
        ('NDSI (Indice de Nieve Normalizado)', 'NDSI'),
        ('BSI (Indice de Suelo Desnudo)', 'BSI'),
        ('SAVI (Indice de Vegetacion Ajustado al Suelo)', 'SAVI'),
        ('NDMI (Indice de Humedad Normalizado)', 'NDMI'),
        ('ELEVACION (SRTM 30 m, metros)', 'ELEVACION'),
        ('PENDIENTE (SRTM, grados)', 'PENDIENTE'),
    ],
    value='MNDWI', description='Estudio:', style=estilo, layout=ancho_total)

figura_selector = widgets.Dropdown(
    options=[('Sin figuras', -1)], value=-1,
    description='Zona activa:', style=estilo, layout=ancho_total)

btn_procesar = boton('Procesar Zona', 'success')
btn_limpiar = boton('Limpiar Todo', 'danger')

# --- Zonas: marcador / figura dibujada ---
input_radio = widgets.FloatText(value=50.0, description='Distancia (m):', style=estilo,
                                layout=ancho_total)
tipo_forma_marker = widgets.Dropdown(
    options=[('Circulo', 'circulo'), ('Cuadrado', 'cuadrado')],
    value='cuadrado', description='Marcador:', style=estilo, layout=ancho_total)

# --- Zonas: estado / municipio / pais ---
nivel_admin = widgets.Dropdown(
    options=[('Pais completo (Mexico)', 'pais'), ('Estado', 'estado'), ('Municipio', 'municipio')],
    value='estado', description='Nivel:', style=estilo, layout=ancho_total)
estado_dd = widgets.Dropdown(options=[], description='Estado:', style=estilo, layout=ancho_total)
municipio_dd = widgets.Dropdown(options=[], description='Municipio:', style=estilo,
                                layout=ancho_total, disabled=True)
btn_admin = boton('Agregar zona', 'primary')
btn_todos_mun = boton('Todos los municipios', 'primary')

# --- OpenStreetMap ---
filtro_osm = widgets.Dropdown(
    options=[('Todo', 'todo'), ('Edificios', 'edificio'), ('Calles', 'calle'),
             ('Agua', 'agua'), ('Servicios', 'servicio'),
             ('Naturaleza / uso de suelo', 'natural'), ('Otras estructuras', 'estructura')],
    value='todo', description='Mostrar:', style=estilo, layout=ancho_total)
btn_cargar_osm = boton('Cargar OSM', 'warning')

# --- Exportar ---
modo_export = widgets.Dropdown(
    options=[('Pixeles (CSV)', 'pixeles'), ('Estadisticas por zona (CSV)', 'estadisticas')],
    value='pixeles', description='Modo:', style=estilo, layout=ancho_total)
input_max_filas = widgets.IntText(value=2000000, description='Max filas/figura:',
                                  style=estilo, layout=ancho_total)
input_error_max = widgets.FloatText(value=0.0, description='Error pos. max (m):',
                                    style=estilo, layout=ancho_total)
btn_previa = boton('Vista previa', 'info')
btn_exportar = boton('Exportar CSVs Originales + Imagenes con Titulo', 'success')

btn_borrar_msg = widgets.Button(description='Borrar mensajes',
                                layout=widgets.Layout(width='130px'))
btn_borrar_msg.on_click(lambda b: salida.clear_output())

# Variables globales de estado
control_leyenda_actual = None
control_indicador_actual = None
historial_geometrias = []
capas_agregadas = []
dibujos_registrados = 0
capa_osm = None
osm_features = []
osm_geoms = {}
silenciar_centrado = False


# ============================================================
# 6. INDICES ESPECTRALES Y TOPOGRAFIA
# ============================================================
def obtener_topografia(estudio):
    if estudio == 'ELEVACION':
        img = DEM.select('elevation').rename('ELEVACION')
        vis = {'min': 0, 'max': 3500,
               'palette': ['#006837', '#78c679', '#ffffbf', '#d95f0e', '#ffffff']}
        etiquetas = ['3500 m', '2600 m', '1750 m', '900 m', '0 m']
    else:
        img = ee.Terrain.slope(DEM).rename('PENDIENTE')
        vis = {'min': 0, 'max': 45,
               'palette': ['#ffffcc', '#a1dab4', '#41b6c4', '#2c7fb8', '#253494']}
        etiquetas = ['45 grados', '34', '22', '11', '0']
    return img, vis, estudio, etiquetas


def obtener_imagen_indice(estudio, region=None):
    if estudio in BANDAS_TOPO:
        return obtener_topografia(estudio)

    base = construir_base(region)
    b = base.select
    if estudio == 'NDVI':
        img = base.normalizedDifference(['B8', 'B4']).rename('NDVI')
        vis = {'min': 0.0, 'max': 0.8, 'palette': ['#8c510a', '#dfc27d', '#f5f5f5', '#80cdc1', '#01665e']}
        etiquetas = ['Muy Densa (0.8)', 'Densa (0.6)', 'Moderada (0.4)', 'Baja (0.2)', 'Suelo/Escaso (0.0)']
    elif estudio == 'NDWI':
        img = base.normalizedDifference(['B3', 'B8']).rename('NDWI')
        vis = {'min': -0.3, 'max': 0.5, 'palette': ['#d73027', '#fee08b', '#ffffbf', '#91bfdb', '#4575b4']}
        etiquetas = ['Agua Alta (0.5)', 'Agua Media (0.3)', 'Humedad (0.0)', 'Transicion (-0.1)', 'Suelo Seco (-0.3)']
    elif estudio == 'MNDWI':
        img = base.normalizedDifference(['B3', 'B11']).rename('MNDWI')
        vis = {'min': -0.5, 'max': 0.4, 'palette': ['#a50026', '#fdae61', '#ffffbf', '#abd9e9', '#08519c']}
        etiquetas = ['Agua Libre (0.4)', 'Agua Somera (0.2)', 'Transicion (0.0)', 'Suelo (-0.2)', 'Urbano / Roca (-0.5)']
    elif estudio == 'EVI':
        img = base.expression(
            '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
            {'NIR': b('B8'), 'RED': b('B4'), 'BLUE': b('B2')}).rename('EVI')
        vis = {'min': 0.0, 'max': 1.0, 'palette': ['#f7fcb9', '#addd8e', '#78c679', '#31a354', '#006837']}
        etiquetas = ['Muy Alta (1.0)', 'Alta (0.75)', 'Media (0.5)', 'Baja (0.25)', 'Minima (0.0)']
    elif estudio == 'NDSI':
        img = base.normalizedDifference(['B3', 'B11']).rename('NDSI')
        vis = {'min': 0.0, 'max': 0.8, 'palette': ['#000000', '#c6dbef', '#9ecae1', '#4292c6', '#ffffff']}
        etiquetas = ['Nieve Alta (0.8)', 'Media (0.6)', 'Baja (0.4)', 'Minima (0.2)', 'Sin Nieve (0.0)']
    elif estudio == 'BSI':
        img = base.expression(
            '((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))',
            {'SWIR': b('B11'), 'RED': b('B4'), 'NIR': b('B8'), 'BLUE': b('B2')}).rename('BSI')
        vis = {'min': -0.4, 'max': 0.4, 'palette': ['#3288bd', '#abdda4', '#ffffbf', '#fdae61', '#d53e4f']}
        etiquetas = ['Suelo Denso (0.4)', 'Alto (0.2)', 'Neutro (0.0)', 'Bajo (-0.2)', 'Vegetacion/Agua (-0.4)']
    elif estudio == 'SAVI':
        img = base.expression(
            '((NIR - RED) / (NIR + RED + 0.5)) * 1.5',
            {'NIR': b('B8'), 'RED': b('B4')}).rename('SAVI')
        vis = {'min': 0.0, 'max': 0.8, 'palette': ['#ffffcc', '#a1dab4', '#41b6c4', '#225ea8', '#0c2c84']}
        etiquetas = ['Alta (0.8)', 'Media-Alta (0.6)', 'Media (0.4)', 'Baja (0.2)', 'Minima (0.0)']
    else:  # NDMI
        img = base.normalizedDifference(['B8', 'B11']).rename('NDMI')
        vis = {'min': -0.4, 'max': 0.4, 'palette': ['#d73027', '#fc8d59', '#fee08b', '#91bfdb', '#4575b4']}
        etiquetas = ['Alta Humedad (0.4)', 'Media-Alta (0.2)', 'Moderada (0.0)', 'Baja (-0.2)', 'Muy Seco (-0.4)']
    return img, vis, estudio, etiquetas


def imagen_todas_bandas(region=None):
    """Una sola imagen con los 8 indices + elevacion + pendiente."""
    img = obtener_imagen_indice(INDICES_ESPECTRALES[0], region)[0]
    for nombre in INDICES_ESPECTRALES[1:]:
        img = img.addBands(obtener_imagen_indice(nombre, region)[0])
    for nombre in BANDAS_TOPO:
        img = img.addBands(obtener_topografia(nombre)[0])
    return img.select(TODAS_LAS_BANDAS)


# ============================================================
# 7. FUNCIONES AUXILIARES DE INTERFAZ
# ============================================================
def refrescar_selector(seleccionar_ultima=True):
    if historial_geometrias:
        figura_selector.options = [(item['nombre'], i) for i, item in enumerate(historial_geometrias)]
        if seleccionar_ultima:
            figura_selector.value = len(historial_geometrias) - 1
    else:
        figura_selector.options = [('Sin figuras', -1)]
        figura_selector.value = -1


def nombre_unico(base):
    usados = {item['nombre'] for item in historial_geometrias}
    if base not in usados:
        return base
    n = 2
    while f"{base} ({n})" in usados:
        n += 1
    return f"{base} ({n})"


def actualizar_indicador_activo(nombre_zona):
    global control_indicador_actual
    if control_indicador_actual is not None:
        try:
            Map.remove_control(control_indicador_actual)
        except Exception:
            pass

    html = f"""
    <div style="background: rgba(255,255,255,0.95); padding: 8px 15px; border-radius: 8px;
                box-shadow: 0 0 15px rgba(0,0,0,0.3); font-family: sans-serif; font-size: 12px;
                font-weight: bold; color: #007bff; text-align: center;">
        {nombre_zona}
    </div>"""
    control_indicador_actual = WidgetControl(widget=widgets.HTML(value=html), position='topright')
    Map.add_control(control_indicador_actual)


def mostrar_leyenda(titulo_leyenda, vis, etiquetas):
    global control_leyenda_actual
    if control_leyenda_actual is not None:
        try:
            Map.remove_control(control_leyenda_actual)
        except Exception:
            pass

    p = vis['palette']
    html = f"""
    <div style="background: white; padding: 10px; border-radius: 5px; box-shadow: 0 0 15px rgba(0,0,0,0.2);
                font-family: sans-serif; font-size: 10px; width: 230px;">
        <b>{titulo_leyenda}</b><br>
        <div style="background: linear-gradient(to top, {p[0]}, {p[1]}, {p[2]}, {p[3]}, {p[4]});
                    width: 18px; height: 120px; float: left; margin-right: 8px; margin-top: 5px;"></div>
        <div style="height: 120px; display: inline-block; line-height: 24px;">
            {'<br>'.join(etiquetas)}
        </div>
    </div>"""
    control_leyenda_actual = WidgetControl(widget=widgets.HTML(value=html), position='bottomleft')
    Map.add_control(control_leyenda_actual)


def obtener_etiqueta_ubicacion(item):
    nombre = item['nombre']
    if item['origen'] == 'admin':
        return nombre.replace("Estado: ", "").replace("Municipio: ", "").replace("Mexico (pais completo)", "Mexico")
    if item['origen'] == 'osm':
        return f"{nombre} (OSM)"
    try:
        coords = item['geometry'].centroid(100).coordinates().getInfo()
        lon, lat = round(coords[0], 4), round(coords[1], 4)
        return f"Ubicacion_[{lat}_{lon}]"
    except Exception:
        return nombre


# ============================================================
# 8. NOMBRES Y CATEGORIAS DE LAS ESTRUCTURAS DE OPENSTREETMAP
# ============================================================
TIPOS_AMENITY = {
    'hospital': 'Hospital', 'clinic': 'Clinica', 'doctors': 'Consultorio',
    'pharmacy': 'Farmacia', 'school': 'Escuela', 'university': 'Universidad',
    'college': 'Colegio', 'kindergarten': 'Kinder', 'library': 'Biblioteca',
    'restaurant': 'Restaurante', 'cafe': 'Cafeteria', 'bank': 'Banco',
    'fuel': 'Gasolinera', 'parking': 'Estacionamiento', 'police': 'Policia',
    'fire_station': 'Bomberos', 'townhall': 'Presidencia municipal',
    'place_of_worship': 'Iglesia/Templo', 'marketplace': 'Mercado',
    'cinema': 'Cine', 'theatre': 'Teatro', 'bus_station': 'Central de autobuses',
}
TIPOS_NATURAL = {
    'water': 'Cuerpo de agua', 'wetland': 'Humedal', 'wood': 'Bosque',
    'scrub': 'Matorral', 'grassland': 'Pastizal', 'bare_rock': 'Roca expuesta',
    'sand': 'Arena', 'beach': 'Playa', 'peak': 'Cerro', 'cliff': 'Acantilado',
}
TIPOS_WATER = {
    'lake': 'Lago', 'pond': 'Estanque', 'reservoir': 'Presa/Embalse',
    'river': 'Rio', 'canal': 'Canal', 'basin': 'Cuenca', 'stream_pool': 'Poza',
}
TIPOS_WATERWAY = {
    'river': 'Rio', 'stream': 'Arroyo', 'canal': 'Canal',
    'drain': 'Drenaje', 'ditch': 'Zanja', 'dam': 'Cortina de presa',
}
TIPOS_LANDUSE = {
    'residential': 'Zona residencial', 'commercial': 'Zona comercial',
    'industrial': 'Zona industrial', 'farmland': 'Terreno agricola',
    'farmyard': 'Granja', 'forest': 'Bosque', 'grass': 'Area de pasto',
    'meadow': 'Pradera', 'orchard': 'Huerto', 'cemetery': 'Panteon',
    'retail': 'Zona de comercios', 'quarry': 'Cantera', 'reservoir': 'Presa/Embalse',
    'military': 'Zona militar', 'recreation_ground': 'Area recreativa',
}
TIPOS_LEISURE = {
    'park': 'Parque', 'garden': 'Jardin', 'pitch': 'Cancha',
    'stadium': 'Estadio', 'playground': 'Juegos infantiles',
    'sports_centre': 'Centro deportivo', 'golf_course': 'Campo de golf',
}
TIPOS_BUILDING = {
    'house': 'Casa', 'apartments': 'Departamentos', 'commercial': 'Edificio comercial',
    'industrial': 'Nave industrial', 'church': 'Iglesia', 'school': 'Escuela',
    'hospital': 'Hospital', 'warehouse': 'Bodega', 'retail': 'Comercio',
}
TIPOS_MANMADE = {
    'bridge': 'Puente', 'tower': 'Torre', 'water_tower': 'Torre de agua',
    'storage_tank': 'Tanque', 'pipeline': 'Tuberia', 'wastewater_plant': 'Planta de aguas residuales',
    'water_works': 'Planta de agua', 'chimney': 'Chimenea',
}
TIPOS_RAILWAY = {'rail': 'Via de tren', 'station': 'Estacion de tren', 'platform': 'Anden'}

ESTILOS = {
    'edificio':    {'color': '#8b5a2b', 'fillColor': '#c68642', 'weight': 1, 'fillOpacity': 0.6},
    'calle':        {'color': '#f28e1c', 'fillColor': '#f28e1c', 'weight': 3, 'fillOpacity': 0.3},
    'agua':         {'color': '#1f78b4', 'fillColor': '#5fa8e8', 'weight': 2, 'fillOpacity': 0.6},
    'servicio':     {'color': '#c0392b', 'fillColor': '#e74c3c', 'weight': 2, 'fillOpacity': 0.55},
    'natural':      {'color': '#3c8d40', 'fillColor': '#7fc97f', 'weight': 1, 'fillOpacity': 0.4},
    'estructura': {'color': '#555555', 'fillColor': '#999999', 'weight': 2, 'fillOpacity': 0.5},
}


def valor(fila, columna):
    v = fila.get(columna)
    if v is None or (not isinstance(v, str) and pd.isna(v)):
        return None
    return str(v)


def describir_osm(fila):
    nombre = valor(fila, 'name')
    for col, tabla, defecto in [
        ('amenity', TIPOS_AMENITY, 'Servicio'),
        ('highway', {}, 'Calle'),
        ('waterway', TIPOS_WATERWAY, 'Cauce'),
        ('water', TIPOS_WATER, 'Cuerpo de agua'),
        ('natural', TIPOS_NATURAL, 'Zona natural'),
        ('building', TIPOS_BUILDING, 'Edificio'),
        ('man_made', TIPOS_MANMADE, 'Estructura'),
        ('railway', TIPOS_RAILWAY, 'Via de tren'),
        ('leisure', TIPOS_LEISURE, 'Area recreativa'),
        ('landuse', TIPOS_LANDUSE, 'Uso de suelo'),
    ]:
        v = valor(fila, col)
        if v is None:
            continue
        if col == 'building' and v == 'yes':
            return 'Edificio', nombre
        return tabla.get(v, defecto), nombre
    return 'Zona', nombre


def categoria_osm(fila):
    if valor(fila, 'amenity'):
        return 'servicio'
    if valor(fila, 'highway'):
        return 'calle'
    if valor(fila, 'waterway') or valor(fila, 'water') or valor(fila, 'natural') in ('water', 'wetland'):
        return 'agua'
    if valor(fila, 'building'):
        return 'edificio'
    if valor(fila, 'man_made') or valor(fila, 'railway'):
        return 'estructura'
    return 'natural'


def geometria_a_poligono(geom, grande=False):
    if geom.geom_type in ('LineString', 'MultiLineString'):
        return geom.buffer(0.00015)
    if geom.geom_type in ('Point', 'MultiPoint'):
        return geom.buffer(0.0003 if grande else 0.0001)
    return geom


# ============================================================
# 9. REGISTRAR LOS DIBUJOS DEL MAPA EN LA LISTA
# ============================================================
def dibujo_a_geometria(dibujo):
    if hasattr(dibujo, 'geometry'):
        return dibujo.geometry()
    if isinstance(dibujo, dict):
        return ee.Geometry(dibujo.get('geometry', dibujo))
    return ee.Feature(dibujo).geometry()


def registrar_dibujos_nuevos():
    global dibujos_registrados
    if not hasattr(Map, 'draw_features'):
        return

    distancia_m = input_radio.value
    forma = tipo_forma_marker.value

    for n in range(dibujos_registrados, len(Map.draw_features)):
        try:
            g = dibujo_a_geometria(Map.draw_features[n])
            tipo_g = g.type().getInfo()

            if tipo_g == 'Point':
                lon, lat = g.coordinates().getInfo()[:2]
                if forma == 'cuadrado':
                    d_lat = distancia_m / 111320.0
                    d_lon = distancia_m / (111320.0 * math.cos(math.radians(lat)))
                    geom = ee.Geometry.BBox(lon - d_lon, lat - d_lat, lon + d_lon, lat + d_lat)
                    nombre = f"Dibujo {n + 1}: Cuadrado de {int(distancia_m * 2)} m"
                else:
                    geom = g.buffer(distancia_m)
                    nombre = f"Dibujo {n + 1}: Circulo de {int(distancia_m)} m de radio"
            else:
                geom = g
                nombres_tipo = {'Polygon': 'Poligono', 'LineString': 'Linea', 'Rectangle': 'Rectangulo'}
                nombre = f"Dibujo {n + 1}: {nombres_tipo.get(tipo_g, tipo_g)}"

            historial_geometrias.append({'geometry': geom, 'nombre': nombre_unico(nombre),
                                         'origen': 'dibujo', 'osm_id': None})
        except Exception as e:
            msg(f"No se pudo leer el dibujo {n + 1}: {e}")

    if len(Map.draw_features) > dibujos_registrados:
        dibujos_registrados = len(Map.draw_features)
        refrescar_selector()


# ============================================================
# 10. ZONAS ADMINISTRATIVAS 
# ============================================================
def cargar_estados():
    try:
        fc = GAUL_ESTADOS.filter(ee.Filter.eq('ADM0_NAME', 'Mexico'))
        estados = sorted(fc.aggregate_array('ADM1_NAME').getInfo())
        estado_dd.options = estados
        if 'Zacatecas' in estados:
            estado_dd.value = 'Zacatecas'
    except Exception as ex:
        msg(f"No se pudo cargar la lista de estados: {ex}")


def _municipios_del_estado():
    return (GAUL_MUNICIPIOS.filter(ee.Filter.eq('ADM0_NAME', 'Mexico'))
            .filter(ee.Filter.eq('ADM1_NAME', estado_dd.value)))


@seguro
def cargar_municipios(*args):
    if not estado_dd.value:
        return
    municipio_dd.options = sorted(_municipios_del_estado().aggregate_array('ADM2_NAME').getInfo())


@seguro
def al_cambiar_nivel(*args):
    estado_dd.disabled = nivel_admin.value == 'pais'
    municipio_dd.disabled = nivel_admin.value != 'municipio'


@seguro
def agregar_zona_admin(b):
    nivel = nivel_admin.value
    if nivel == 'pais':
        f = GAUL_PAISES.filter(ee.Filter.eq('ADM0_NAME', 'Mexico')).first()
        geom = ee.Feature(f).geometry().simplify(100)
        nombre = 'Mexico (pais completo)'
    elif nivel == 'estado':
        fc = (GAUL_ESTADOS.filter(ee.Filter.eq('ADM0_NAME', 'Mexico'))
               .filter(ee.Filter.eq('ADM1_NAME', estado_dd.value)))
        geom = fc.geometry().dissolve().simplify(100)
        nombre = f"Estado: {estado_dd.value}"
    else:
        f = _municipios_del_estado().filter(ee.Filter.eq('ADM2_NAME', municipio_dd.value)).first()
        geom = ee.Feature(f).geometry()
        nombre = f"Municipio: {municipio_dd.value}, {estado_dd.value}"

    historial_geometrias.append({'geometry': geom, 'nombre': nombre_unico(nombre),
                                 'origen': 'admin', 'osm_id': None})
    refrescar_selector()
    msg(f"Zona agregada: {nombre}. Ya puedes 'Procesar Zona' o exportar.")


@seguro
def agregar_todos_municipios(b):
    if not estado_dd.value:
        msg("Elige un estado primero.")
        return
    mun = _municipios_del_estado()
    nombres = mun.aggregate_array('ADM2_NAME').getInfo()
    if len(nombres) > MAX_MUNICIPIOS:
        msg(f"{len(nombres)} municipios; el maximo es {MAX_MUNICIPIOS}.")
        return
    lista = mun.toList(len(nombres))
    for i, nom in enumerate(nombres):
        historial_geometrias.append({
            'geometry': ee.Feature(lista.get(i)).geometry(),
            'nombre': nombre_unico(f"Municipio: {nom}, {estado_dd.value}"),
            'origen': 'admin', 'osm_id': None})
    refrescar_selector()
    msg(f"{len(nombres)} municipios agregados. Para exportarlos usa 'Estadisticas por zona'.")


estado_dd.observe(cargar_municipios, names='value')
nivel_admin.observe(al_cambiar_nivel, names='value')
btn_admin.on_click(agregar_zona_admin)
btn_todos_mun.on_click(agregar_todos_municipios)


# ============================================================
# 11. ESTRUCTURAS DE OPENSTREETMAP DIBUJADAS EN EL MAPA
# ============================================================
ox.settings.timeout = 300
ox.settings.overpass_endpoint = "https://overpass-api.de/api/interpreter"


def obtener_area_estudio():
    registrar_dibujos_nuevos()

    dibujos = [it for it in historial_geometrias if it['origen'] == 'dibujo']
    i = figura_selector.value
    if 0 <= i < len(historial_geometrias) and historial_geometrias[i]['origen'] == 'dibujo':
        elegido = historial_geometrias[i]
    elif dibujos:
        elegido = dibujos[-1]
    else:
        elegido = None

    if elegido is not None:
        poligono = geometria_a_poligono(shape(elegido['geometry'].getInfo()))
        return poligono, elegido['nombre']

    limites = Map.bounds
    if limites and len(limites) == 2:
        (s, w), (n, e) = limites
        return box(w, s, e, n), 'area visible del mapa'
    return None, None


def estimar_km2(poligono):
    minx, miny, maxx, maxy = poligono.bounds
    lat = (miny + maxy) / 2
    ancho = (maxx - minx) * 111.32 * math.cos(math.radians(lat))
    alto = (maxy - miny) * 111.32
    return ancho * alto


def estilo_estructura(feature):
    return ESTILOS.get(feature['properties']['categoria'], ESTILOS['estructura'])


@seguro
def al_clic_estructura(event=None, feature=None, **kwargs):
    global silenciar_centrado
    if feature is None:
        return
    props = feature['properties']
    osm_id = props['id']

    silenciar_centrado = True
    try:
        for idx, item in enumerate(historial_geometrias):
            if item['osm_id'] == osm_id:
                figura_selector.value = idx
                actualizar_indicador_activo(item['nombre'])
                return

        geom = geometria_a_poligono(osm_geoms[osm_id], grande=True)
        historial_geometrias.append({
            'geometry': ee.Geometry(mapping(geom)),
            'nombre': nombre_unico(props['etiqueta']),
            'origen': 'osm',
            'osm_id': osm_id,
        })
        refrescar_selector()
        actualizar_indicador_activo(historial_geometrias[-1]['nombre'])
    finally:
        silenciar_centrado = False


@seguro
def dibujar_capa_osm(*args):
    global capa_osm
    if capa_osm is not None:
        try:
            Map.remove_layer(capa_osm)
        except Exception:
            pass
        capa_osm = None

    filtro = filtro_osm.value
    seleccion = [f for f in osm_features
                 if filtro == 'todo' or f['properties']['categoria'] == filtro]
    if not seleccion:
        return
    if len(seleccion) > MAX_DIBUJO:
        msg(f"Hay {len(seleccion)} estructuras; se dibujan las primeras {MAX_DIBUJO}. "
            "Usa el filtro 'Mostrar' para ver el resto.")
        seleccion = seleccion[:MAX_DIBUJO]

    coleccion = {'type': 'FeatureCollection', 'features': seleccion}
    capa_osm = GeoJSON(
        data=coleccion,
        style_callback=estilo_estructura,
        hover_style={'weight': 4, 'color': '#ffff00', 'fillOpacity': 0.8},
        name='Estructuras OSM')
    capa_osm.on_click(al_clic_estructura)
    Map.add_layer(capa_osm)


filtro_osm.observe(dibujar_capa_osm, names='value')


@seguro
def cargar_datos_osm(b):
    global osm_features, osm_geoms
    msg("Buscando estructuras (edificios, calles, servicios) en OpenStreetMap...")
    poligono, origen = obtener_area_estudio()
    if poligono is None:
        msg("Dibuja una figura (o mueve el mapa) antes de cargar OSM.")
        return

    km2 = estimar_km2(poligono)
    if km2 > MAX_KM2:
        msg(f"El area es de {km2:.1f} km2 (maximo para OSM: {MAX_KM2} km2). "
            "Dibuja una zona mas pequena.")
        return

    msg(f"Area: {origen} (~{km2:.2f} km2). Descargando, puede tardar unos segundos...")

    tags_interes = {
        'amenity': True, 'highway': True, 'waterway': True, 'water': True,
        'natural': True, 'landuse': True, 'leisure': True, 'building': True,
        'man_made': True, 'railway': True,
    }

    intentos = 4
    espera = 5
    gdf = None
    endpoints = [
        "https://overpass-api.de/api/interpreter",
        "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
        "https://overpass.kumi.systems/api/interpreter",
    ]

    for intento in range(intentos):
        try:
            ox.settings.overpass_endpoint = endpoints[intento % len(endpoints)]
            gdf = ox.features_from_polygon(poligono, tags=tags_interes)
            if not gdf.empty:
                break
        except Exception as ex:
            msg(f"Intento {intento + 1} fallido: {str(ex)[:200]}")
            if intento < intentos - 1:
                msg(f"Esperando {espera} segundos y cambiando de espejo Overpass...")
                time.sleep(espera)
                espera *= 2
            else:
                msg("Error: los servidores de OpenStreetMap estan saturados.")
                return

    if gdf is None or gdf.empty:
        msg("No se encontraron estructuras en esta area.")
        return

    osm_features = []
    osm_geoms = {}
    for n, (_, fila) in enumerate(gdf.iterrows()):
        geom = fila.geometry
        if geom is None or geom.is_empty:
            continue

        tipo, nombre = describir_osm(fila)
        etiqueta = f"{tipo}: {nombre}" if nombre else f"{tipo} (sin nombre)"
        osm_geoms[n] = geom

        osm_features.append({
            'type': 'Feature',
            'geometry': json.loads(json.dumps(mapping(geometria_a_poligono(geom)))),
            'properties': {'id': n, 'tipo': tipo, 'nombre': nombre or '',
                           'categoria': categoria_osm(fila), 'etiqueta': etiqueta},
        })

    dibujar_capa_osm()

    resumen = {}
    for f in osm_features:
        c = f['properties']['categoria']
        resumen[c] = resumen.get(c, 0) + 1
    detalle = ', '.join(f"{k}: {v}" for k, v in sorted(resumen.items()))
    msg(f"Listo: {len(osm_features)} estructuras cargadas ({detalle}).")
    msg("Haz clic en una para seleccionarla y luego presiona 'Procesar Zona'.")


btn_cargar_osm.on_click(cargar_datos_osm)


# ============================================================
# 12. PROCESAR LA ZONA SELECCIONADA
# ============================================================
@seguro
def ejecutar_analisis_individual(b):
    estudio = index_selector.value
    registrar_dibujos_nuevos()

    i = figura_selector.value
    if not (0 <= i < len(historial_geometrias)):
        msg("Atencion: dibuja una figura, agrega un estado/municipio o haz clic en una estructura.")
        return

    item = historial_geometrias[i]
    geometria = item['geometry']
    nombre_zona = item['nombre']

    capa_previa = item.get('capa')
    if capa_previa is not None:
        try:
            Map.remove_layer(capa_previa)
        except Exception:
            pass
        if capa_previa in capas_agregadas:
            capas_agregadas.remove(capa_previa)

    msg(f"Procesando '{nombre_zona}' ({estudio})...")
    layer_img, vis, titulo_ind, etiquetas = obtener_imagen_indice(estudio, geometria)
    nombre_capa = f"{nombre_zona} - {estudio}"

    Map.addLayer(layer_img.clip(geometria), vis, nombre_capa)
    item['capa'] = Map.layers[-1]
    item['indice_mostrado'] = estudio
    capas_agregadas.append(item['capa'])

    mostrar_leyenda(titulo_ind, vis, etiquetas)
    actualizar_indicador_activo(nombre_zona)
    try:
        Map.center_object(geometria)
    except Exception:
        pass

    msg(f"Listo: '{nombre_zona}' con el indice {titulo_ind}.")


btn_procesar.on_click(ejecutar_analisis_individual)


# ============================================================
# 13. CAMBIAR DE ZONA ACTIVA
# ============================================================
@seguro
def al_cambiar_figura(change):
    i = change['new']
    if isinstance(i, int) and 0 <= i < len(historial_geometrias):
        item = historial_geometrias[i]
        actualizar_indicador_activo(item['nombre'])
        if silenciar_centrado:
            return
        try:
            Map.center_object(item['geometry'])
        except Exception:
            pass


figura_selector.observe(al_cambiar_figura, names='value')


# ============================================================
# 14. EXPORTAR A GOOGLE DRIVE (CSV original + Imagen con Título y Leyenda)
# ============================================================
def escala_para(area_m2, max_filas):
    ideal = math.sqrt(max(area_m2, 1.0) / max(max_filas, 1))
    for e in ESCALAS:
        if e >= ideal:
            return e
    return ESCALAS[-1]


def planificar_exportacion():
    registrar_dibujos_nuevos()
    items = list(historial_geometrias)
    if not items:
        return None

    max_filas = max(1000, int(input_max_filas.value))
    err_max = float(input_error_max.value)
    escala_tope = None
    if err_max > 0:
        validas = [e for e in ESCALAS if e * FACTOR_ERROR_POS <= err_max]
        escala_tope = max(validas) if validas else ESCALAS[0]

    areas = ee.List([it['geometry'].area(1000) for it in items]).getInfo()

    plan = []
    for i, (it, a) in enumerate(zip(items, areas)):
        esc = escala_para(a, max_filas)
        limitada = False
        if escala_tope is not None and esc > escala_tope:
            esc, limitada = escala_tope, True
        plan.append({'i': i, 'nombre': it['nombre'], 'area_km2': a / 1e6, 'escala': esc,
                     'error_m': round(esc * FACTOR_ERROR_POS, 1),
                     'filas': a / (esc ** 2), 'limitada': limitada})
    return items, plan


def mostrar_plan(plan, modo):
    msg(f"--- Plan de exportacion: {len(plan)} figura(s), modo '{modo}' ---")
    for p in plan:
        msg(f" - {p['nombre']}: Escala automatica {p['escala']} m (~{p['filas']:,.0f} filas)")


@seguro
def vista_previa_exportacion(b):
    r = planificar_exportacion()
    if r is None:
        msg("Primero dibuja o agrega una zona.")
        return
    mostrar_plan(r[1], modo_export.value)


def agregar_error_estandar(f):
    props = {}
    for b in TODAS_LAS_BANDAS:
        sd = ee.Number(ee.Algorithms.If(f.get(b + '_stdDev'), f.get(b + '_stdDev'), 0))
        n = ee.Number(ee.Algorithms.If(f.get(b + '_count'), f.get(b + '_count'), 0))
        props[b + '_error_est'] = ee.Algorithms.If(n.gt(1), sd.divide(n.sqrt()), -9999)
    return f.set(props)


@seguro
def exportar_pixeles_zona(b):
    r = planificar_exportacion()
    if r is None:
        msg("Atencion: primero dibuja o agrega una zona.")
        return
    items, plan = r
    modo = modo_export.value
    estudio_actual = index_selector.value
    mostrar_plan(plan, modo)

    sello = time.strftime('%Y%m%d_%H%M%S')
    rango_fechas = f"{FECHA_INI} al {FECHA_FIN}"

    # 1. Recorrer y generar la imagen PNG a color para CADA UNA de las figuras/zonas registradas
    for p in plan:
        item = items[p['i']]
        geom = item['geometry']
        estudio_zona = item.get('indice_mostrado', estudio_actual)
        ubicacion_limpia = obtener_etiqueta_ubicacion(item)

        try:
            img_ind, vis, titulo_ind, etiquetas = obtener_imagen_indice(estudio_zona, geom)
            
            # Visualizar con la paleta de colores de GEE
            img_rgb = img_ind.clip(geom).visualize(**vis)
            
            # Obtener la URL del thumbnail directamente de GEE (método 100% robusto para todas las figuras)
            url_thumb = img_rgb.getThumbURL({
                'region': geom,
                'dimensions': '800x800',
                'format': 'png'
            })
            
            # Descargar la imagen procesada de forma segura
            import urllib.request
            nombre_png = f"mapa_{estudio_zona}_{ubicacion_limpia}_{sello}.png".replace(" ", "_").replace(",", "")
            
            # Usar Matplotlib para incrustar el título profesional y la barra de colores oficial
            import matplotlib.pyplot as plt
            from PIL import Image
            import io
            
            with urllib.request.urlopen(url_thumb) as respuesta:
                img_bytes = respuesta.read()
            
            imagen_pil = Image.open(io.BytesIO(img_bytes))
            
            fig, ax = plt.subplots(figsize=(8, 8), dpi=300)
            ax.set_title(f"Indicador: {estudio_zona}\nZona: {ubicacion_limpia}\nPeriodo: {rango_fechas}",
                         fontsize=12, fontweight='bold', pad=15)
            
            ax.imshow(imagen_pil)
            ax.axis('off')
            
            # Añadir barra de color horizontal (colorbar) limpia y profesional
            from matplotlib.colors import LinearSegmentedColormap
            cmap = LinearSegmentedColormap.from_list("gee_palette", vis['palette'])
            sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vis['min'], vmax=vis['max']))
            sm.set_array([])
            
            cbar = fig.colorbar(sm, ax=ax, orientation='horizontal', pad=0.08, shrink=0.7)
            cbar.set_label(f"Valores del indicador ({estudio_zona}): min {vis['min']} / max {vis['max']}", fontsize=10, fontweight='bold')
            
            plt.tight_layout()
            plt.savefig(nombre_png, bbox_inches='tight', facecolor='white')
            plt.close()
            msg(f"¡Imagen PNG generada con éxito para la zona: {ubicacion_limpia}!")
            
        except Exception as ex:
            msg(f"Aviso al generar imagen para {ubicacion_limpia}: {ex}")

    # 2. Exportar los CSVs limpios (sin la primera columna 'estudio_analizado')
    grupos = {}
    for p in plan:
        grupos.setdefault(p['escala'], []).append(p)

    for esc, lista in sorted(grupos.items()):
        feats = []
        for p in lista:
            item = items[p['i']]
            estudio_zona = item.get('indice_mostrado', estudio_actual)
            feats.append(ee.Feature(item['geometry'], {
                'estudio_analizado': estudio_zona,
                'id_zona': p['i'],
                'escala_m': esc,
                'error_posicion_max_m': p['error_m']
            }))

        figuras = ee.FeatureCollection(feats)
        imagen = imagen_todas_bandas(figuras.geometry(1).bounds())
        base_cols = ['id_zona', 'escala_m', 'error_posicion_max_m']

        if modo == 'pixeles':
            columnas = base_cols + TODAS_LAS_BANDAS + ['longitude', 'latitude']
            coleccion = imagen.addBands(ee.Image.pixelLonLat()).sampleRegions(
                collection=figuras, properties=base_cols,
                scale=esc, tileScale=16, geometries=False)
            prefijo = f'pixeles_{estudio_actual}_{sello}_{esc}m'
        else:
            reductor = (ee.Reducer.mean()
                        .combine(ee.Reducer.stdDev(), sharedInputs=True)
                        .combine(ee.Reducer.minMax(), sharedInputs=True)
                        .combine(ee.Reducer.count(), sharedInputs=True))
            columnas = base_cols + [f"{b}_{s}" for b in TODAS_LAS_BANDAS
                                    for s in ('mean', 'error_est', 'stdDev', 'min', 'max', 'count')]
            coleccion = imagen.reduceRegions(
                collection=figuras, reducer=reductor, scale=esc, tileScale=16
            ).map(agregar_error_estandar)
            prefijo = f'estadisticas_{estudio_actual}_{sello}_{esc}m'

        tarea = ee.batch.Export.table.toDrive(
            collection=coleccion,
            description=prefijo[:100],
            folder='Estudio_Satelital_CSVs',
            fileFormat='CSV',
            fileNamePrefix=prefijo,
            selectors=columnas
        )
        tarea.start()
        msg(f"Tarea CSV iniciada correctamente para escala {esc}m: {prefijo}.")

    msg("¡Proceso completado! Todas las imágenes PNG y tareas CSV fueron procesadas.")


btn_previa.on_click(vista_previa_exportacion)
btn_exportar.on_click(exportar_pixeles_zona)


# ============================================================
# 15. LIMPIAR TODO
# ============================================================
@seguro
def limpiar_todo_sistema(b):
    global control_leyenda_actual, control_indicador_actual, dibujos_registrados
    global capa_osm, osm_features, osm_geoms

    historial_geometrias.clear()
    refrescar_selector()

    for capa in capas_agregadas:
        try:
            Map.remove_layer(capa)
        except Exception:
            pass
    capas_agregadas.clear()

    if capa_osm is not None:
        try:
            Map.remove_layer(capa_osm)
        except Exception:
            pass
    capa_osm = None
    osm_features = []
    osm_geoms = {}

    try:
        if hasattr(Map, 'draw_control') and Map.draw_control is not None:
            Map.draw_control.clear()
    except Exception:
        pass

    if hasattr(Map, 'draw_features'):
        Map.draw_features.clear()

    dibujos_registrados = 0

    for ctrl in (control_leyenda_actual, control_indicador_actual):
        if ctrl is not None:
            try:
                Map.remove_control(ctrl)
            except Exception:
                pass
    control_leyenda_actual = None
    control_indicador_actual = None

    msg("Sistema limpiado por completo.")


btn_limpiar.on_click(limpiar_todo_sistema)


# ============================================================
# 16. PANEL DE CONTROLES (pestanas)
# ============================================================
def fila(*hijos):
    return widgets.HBox(list(hijos), layout=widgets.Layout(width='100%'))


tab_analisis = widgets.VBox([
    titulo('1. Elige el estudio y la zona'),
    index_selector, figura_selector,
    fila(btn_procesar, btn_limpiar),
])

tab_zonas = widgets.VBox([
    titulo('Marker / figura dibujada en el mapa'),
    input_radio, tipo_forma_marker,
    titulo('Pais, estado o municipio'),
    nivel_admin, estado_dd, municipio_dd,
    fila(btn_admin, btn_todos_mun),
])

tab_osm = widgets.VBox([
    titulo('Estructuras de OpenStreetMap (zonas de hasta 25 km2)'),
    filtro_osm, fila(btn_cargar_osm),
])

tab_exportar = widgets.VBox([
    titulo('Exportar todas las figuras a Google Drive'),
    modo_export, input_max_filas, input_error_max,
    fila(btn_previa, btn_exportar),
])

pestanas = widgets.Tab(children=[tab_analisis, tab_zonas, tab_osm, tab_exportar])
for idx, nombre_tab in enumerate(['Analisis', 'Zonas', 'OSM', 'Exportar']):
    pestanas.set_title(idx, nombre_tab)

cuerpo_panel = widgets.VBox([pestanas, widgets.HBox([titulo('Mensajes'), btn_borrar_msg]), salida],
                            layout=widgets.Layout(width='100%'))

btn_plegar = widgets.ToggleButton(value=False, description='Ocultar panel', icon='chevron-down',
                                  layout=widgets.Layout(width='140px'))


def plegar_panel(change):
    cuerpo_panel.layout.display = 'none' if change['new'] else 'flex'
    btn_plegar.description = 'Mostrar panel' if change['new'] else 'Ocultar panel'
    btn_plegar.icon = 'chevron-up' if change['new'] else 'chevron-down'


btn_plegar.observe(plegar_panel, names='value')

panel_controles = widgets.VBox(
    [widgets.HBox([widgets.HTML("<b style='font-size:14px'>Estudio Satelital</b>"), btn_plegar],
                  layout=widgets.Layout(justify_content='space-between', width='100%')),
     cuerpo_panel],
    layout=widgets.Layout(width='520px', padding='6px'))

control_menu = WidgetControl(widget=panel_controles, position='bottomright')
Map.add_control(control_menu)

# Cargar la lista de estados al final
cargar_estados()

# Mostrar el mapa
Map

MapaSeguro(center=[22.58, -102.52], controls=(WidgetControl(options=['position', 'transparent_bg'], position='…